In [Reading Data](https://lizhen0909.github.io/nu-stat303-1-sec20-coursebook-clean/Reading_data.html), you learned to import a table and check its structure. Now we will use that table to answer questions: **Which records meet a condition? Which columns do we need? How can we calculate and summarize a useful variable?**

This chapter follows a practical sequence:

**Select → filter → sort → create or update → summarize and explain.**

By the end, you should be able to:

- Select a column as a Series or a table as a DataFrame.
- Build Boolean masks and distinguish row labels from row positions.
- Sort records and identify an extreme value or the record containing it.
- Create, rename, update, and remove columns deliberately.
- Convert text and dates while checking values that become missing.
- Calculate interpretable summaries and proportions with explicit denominators.

You do not need NumPy for this chapter. NumPy Fundamentals follows and will develop array shapes, positional operations, and axes.

## Set Up Your Practice Files {#set-up-your-practice-files}

Download the [Pandas Fundamentals practice kit](https://lizhen0909.github.io/nu-stat303-1-sec20-coursebook-clean/downloads/pandas-fundamentals-practice.zip), extract it, and place `stat303-pandas-fundamentals` inside your existing `stat303-setup` project. Select the project environment verified in the setup chapters. The examples need pandas, which you already installed.

```text
stat303-setup/
├── .venv/
└── stat303-pandas-fundamentals/
    ├── pandas_examples.ipynb
    ├── activity04.ipynb
    ├── README.md
    └── data/
        ├── movie_ratings.csv
        ├── Top 10 Albums By Year.csv
        └── STAT303-1 survey for data analysis.csv
```

Use `pandas_examples.ipynb` for worked examples and `activity04.ipynb` for your own report. Both should run with `stat303-pandas-fundamentals` as the notebook's working directory. Keep the input files unchanged.


In [1]:
from pathlib import Path
import pandas as pd

movie_path = Path('data') / 'movie_ratings.csv'
print('Working folder:', Path.cwd().name)
print('Movie file found:', movie_path.is_file())


Working folder: stat303-pandas-fundamentals
Movie file found: True


The check should return `True`. If it does not, inspect `Path.cwd()`, the extracted folder, and the exact filename. See the [Reading Data setup](https://lizhen0909.github.io/nu-stat303-1-sec20-coursebook-clean/Reading_data.html#set-up-the-chapter-files) for the working-directory check. The terminal and notebook can have different current directories.

## A Small Table for Learning Selection {#a-small-table}

A dictionary of lists builds a DataFrame: the keys become column names, and each list supplies a column. The lists here have equal lengths. We deliberately use row labels that differ from row positions.


In [2]:
screenings = pd.DataFrame(
    {
        'Title': ['Cedar', 'Harbor', 'Orbit', 'Meadow', 'Ember'],
        'Rating': [7.2, 8.1, 6.5, 8.1, 7.8],
        'Tickets': [120, 80, 150, 90, 110],
    },
    index=[104, 101, 107, 103, 109],
)
screenings


,Title,Rating,Tickets
104,Cedar,7.2,120
101,Harbor,8.1,80
107,Orbit,6.5,150
103,Meadow,8.1,90
109,Ember,7.8,110


These are invented screenings for illustration. One row is one screening; `Title`, `Rating`, and `Tickets` are variables. The first row's **label** is 104 and its **position** is 0. Position describes where a row currently appears; it is not another stored index column.

For comparison, a list of dictionaries represents one row per dictionary, with dictionary keys identifying columns. A Series can be created from a list or a dictionary of label–value pairs:


In [3]:
ticket_prices = pd.Series({'weekday': 10, 'weekend': 12})
print(ticket_prices)


weekday    10
weekend    12
dtype: int64


The main work in this chapter uses the imported movie DataFrame. Small tables such as `screenings` make selection rules easier to see before applying them to thousands of rows.

## Select Columns and Filter Rows {#select-columns-and-filter-rows}

### One Column Can Be a Series or a DataFrame


In [4]:
rating_series = screenings['Rating']
rating_table = screenings[['Rating']]
print('Single name:', type(rating_series).__name__, rating_series.shape)
print('List of names:', type(rating_table).__name__, rating_table.shape)
rating_table


Single name: Series (5,)
List of names: DataFrame (5, 1)


,Rating
104,7.2
101,8.1
107,6.5
103,8.1
109,7.8


`screenings['Rating']` selects a Series. `screenings[['Rating']]` passes a **list containing one column name** and preserves a two-dimensional DataFrame. The inner brackets construct the list; the outer brackets perform selection.

Prefer bracket notation for columns. It handles names containing spaces, such as `movies['IMDB Rating']`, and avoids collisions with methods such as `count`. With these unique column names, selecting a single name returns a Series; a list of names returns a DataFrame.


In [5]:
screenings[['Title', 'Tickets']]


,Title,Tickets
104,Cedar,120
101,Harbor,80
107,Orbit,150
103,Meadow,90
109,Ember,110


### Build a Mask Before Filtering

A comparison applied to a Series produces a Boolean Series. Each Boolean value answers the question for its corresponding row.


In [6]:
high_rating = screenings['Rating'] >= 8
print(high_rating)
screenings.loc[high_rating, ['Title', 'Rating']]


104    False
101     True
107    False
103     True
109    False
Name: Rating, dtype: bool


,Title,Rating
101,Harbor,8.1
103,Meadow,8.1


The mask retains the row labels. The result contains only rows for which the condition is `True`. Read this as: **keep screenings rated at least 8, and show their titles and ratings**.

### Combine Conditions

Use `&` for elementwise AND, `|` for elementwise OR, and `~` to negate a Boolean mask. Put parentheses around each comparison. Python's `and` and `or` do not combine pandas Series in this way.


In [7]:
selected = (screenings['Rating'] >= 8) & (screenings['Tickets'] < 90)
screenings.loc[selected, ['Title', 'Rating', 'Tickets']]


,Title,Rating,Tickets
101,Harbor,8.1,80


Both conditions must hold for the same row. Replacing `&` with `|` would keep a row satisfying either condition.

For membership in a set of categories, use `.isin()` rather than writing many equality comparisons:


In [8]:
chosen_titles = screenings['Title'].isin(['Harbor', 'Meadow'])
screenings.loc[chosen_titles, ['Title', 'Tickets']]


,Title,Tickets
101,Harbor,80
103,Meadow,90


**Pause and predict:** Which rows would `~chosen_titles` keep? Would it change the original table?

## Distinguish Labels from Positions {#labels-and-positions}

### `.loc` Selects Labels; `.iloc` Selects Positions

Both indexers use the form `[rows, columns]`, but interpret their inputs differently.

| Request | Expression | Meaning |
|---|---|---|
| Rows labeled 101 and 103 | `screenings.loc[[101, 103], :]` | Select these labels, in the requested order |
| Rows at positions 1 and 2 | `screenings.iloc[1:3, :]` | Select positions 1 and 2; stop before 3 |
| All rows, two named columns | `screenings.loc[:, ['Title', 'Rating']]` | Select columns by name |
| First two rows, first two columns | `screenings.iloc[:2, :2]` | Select by position on both axes |

The colon alone means all entries on that axis.


In [9]:
print('Labels 101 and 103:')
print(screenings.loc[[101, 103], ['Title', 'Rating']])
print('\nPositions 1 and 2:')
print(screenings.iloc[1:3, :2])


Labels 101 and 103:
      Title  Rating
101  Harbor     8.1
103  Meadow     8.1

Positions 1 and 2:
      Title  Rating
101  Harbor     8.1
107   Orbit     6.5


### Label Slices Follow the Current Row Order

With the unique labels in this example, a label slice starts at the first endpoint and continues through the second endpoint in the table's current order. Both endpoints are included.


In [10]:
screenings.loc[101:103, ['Title', 'Rating']]


,Title,Rating
101,Harbor,8.1
107,Orbit,6.5
103,Meadow,8.1


This returns labels **101, 107, and 103**, because those rows lie between the two endpoints in the displayed order. It does not mean every integer label numerically between 101 and 103. Missing or duplicated slice endpoints can make label slicing more complicated; use explicit label lists or Boolean masks when those express your intent more clearly.

### Sorting Changes Positions, Not Row Identity


In [11]:
ordered = screenings.sort_values('Rating', ascending=False, kind='stable')
ordered


,Title,Rating,Tickets
101,Harbor,8.1,80
103,Meadow,8.1,90
109,Ember,7.8,110
104,Cedar,7.2,120
107,Orbit,6.5,150


In [12]:
print('First row after sorting:', ordered.iloc[0]['Title'])
print('Row still labeled 104:', ordered.loc[104, 'Title'])


First row after sorting: Harbor
Row still labeled 104: Cedar


The labels move with their records. `.iloc[0]` now refers to the first record in the sorted table, while `.loc[104]` still retrieves Cedar. `kind='stable'` preserves the existing order among ties for this single-column sort.

For filtering, prefer `.loc[mask]`. Although `.iloc` accepts a Boolean array of matching length, it does not accept an indexed Boolean Series directly. You do not need to convert masks to arrays for the work in this chapter.

### Change an Index Deliberately

When a column supplies meaningful row labels, `set_index()` can move it into the index. `reset_index()` normally moves those labels back into a column and supplies a new default index.


In [13]:
by_title = screenings.set_index('Title')
print('New index:', by_title.index.tolist())
restored = by_title.reset_index()
restored.head(2)


New index: ['Cedar', 'Harbor', 'Orbit', 'Meadow', 'Ember']


,Title,Rating,Tickets
0,Cedar,7.2,120
1,Harbor,8.1,80


Here, `screenings` is unchanged because we assigned the returned objects to new names. The original numerical labels are not retained by `set_index('Title')`; preserve them first if they carry needed information. An index need not be unique, so `.loc[label]` is not guaranteed to return just one row in every dataset.

## Sort Movies and Find Extremes {#sort-and-find-extremes}

Return to the historical movie data from Reading Data. We will load it once and use separate names for derived tables.


In [14]:
movies = pd.read_csv(movie_path)
print('Movie table shape:', movies.shape)
movies[['Title', 'IMDB Rating', 'Production Budget']].head(3)


Movie table shape: (2228, 11)


,Title,IMDB Rating,Production Budget
0,Opal Dreams,6.5,9000000
1,Major Dundee,6.7,3800000
2,The Informers,5.2,18000000


### Sort by One or More Variables

**Question:** Which movies have the highest ratings? For tied ratings, show larger vote counts first.


In [15]:
ranked_movies = movies.sort_values(
    ['IMDB Rating', 'IMDB Votes', 'Title'],
    ascending=[False, False, True],
)
ranked_movies[['Title', 'IMDB Rating', 'IMDB Votes']].head(5)


,Title,IMDB Rating,IMDB Votes
182,The Shawshank Redemption,9.2,519541
2084,Inception,9.1,188247
561,The Dark Knight,8.9,465000
1962,Pulp Fiction,8.9,417703
790,Schindler's List,8.9,276283


The sorting keys are applied in order: rating descending, then votes descending within rating ties, then title alphabetically within remaining ties. Sorting preserves existing row labels unless you explicitly request a new index.

`movies.nlargest(5, 'Worldwide Gross')` is a convenient alternative when you only want the largest few values of a numeric variable. The default keeps the first rows encountered at a tied cutoff; `keep='all'` may return more than five rows when the cutoff is tied. State the tie rule when it matters.

### Extreme Value Versus Record Containing It


In [16]:
maximum_gross = movies['Worldwide Gross'].max()
maximum_label = movies['Worldwide Gross'].idxmax()
print('Maximum worldwide gross:', maximum_gross)
print('Label of first maximum:', maximum_label)
movies.loc[maximum_label, ['Title', 'Worldwide Gross']]


Maximum worldwide gross: 2767891499
Label of first maximum: 2094


Title                  Avatar
Worldwide Gross    2767891499
Name: 2094, dtype: object

`.max()` returns the value. `.idxmax()` returns the label of its first occurrence; `.loc` then retrieves the record. This works as a single-record lookup here because the imported index is unique. Ties, duplicate labels, and columns with no observed values need explicit handling.

**Pause and explain:** Why would passing `maximum_label` to `.iloc` be unsafe after sorting?

## Create Columns and Make Reliable Updates {#create-and-update-columns}

### Calculate a Variable with Meaningful Units

**Question:** By how many millions of dollars does worldwide gross exceed the recorded production budget?


In [17]:
movie_analysis = movies.copy()
movie_analysis['gross_minus_budget_millions'] = (
    movie_analysis['Worldwide Gross'] - movie_analysis['Production Budget']
) / 1_000_000
movie_analysis[['Title', 'gross_minus_budget_millions']].head(3)


,Title,gross_minus_budget_millions
0,Opal Dreams,-8.985557
1,Major Dundee,-3.785127
2,The Informers,-17.685000


Pandas performs the arithmetic across corresponding rows without an explicit Python loop. The result is measured in **millions of dollars**. It is not actual profit: the file does not account for all costs or how ticket revenue is distributed.

For a ratio, keep only records with an observed, positive denominator. Do not replace zero budgets with 1 to make division run; that would invent budget information.


In [18]:
positive_budget = movie_analysis['Production Budget'].notna() & (
    movie_analysis['Production Budget'] > 0
)
budget_movies = movie_analysis.loc[positive_budget].copy()
budget_movies['gross_to_budget'] = (
    budget_movies['Worldwide Gross'] / budget_movies['Production Budget']
)
print('Rows excluded for missing or nonpositive budgets:', len(movie_analysis) - len(budget_movies))
budget_movies[['Title', 'gross_to_budget']].head(3)


Rows excluded for missing or nonpositive budgets: 0


,Title,gross_to_budget
0,Opal Dreams,0.001605
1,Major Dundee,0.003914
2,The Informers,0.017500


This dimensionless ratio compares two recorded quantities; it is not a complete return-on-investment measure. Report excluded records when interpreting a derived variable.

### Assign the Return Value When You Want a New Table


In [19]:
renamed = movie_analysis.rename(columns={'IMDB Rating': 'rating'})
print('Original still has IMDB Rating:', 'IMDB Rating' in movie_analysis.columns)
print('Returned table has rating:', 'rating' in renamed.columns)


Original still has IMDB Rating: True
Returned table has rating: True


Methods such as `rename()`, `drop()`, and `sort_values()` return a result by default. Calling them without assignment does not replace the original variable. A clear default is `result = df.method(...)`, or `df = df.method(...)` when you intend to keep the transformed result under the same name.

Use `columns=` when removing columns to make the intent explicit:


In [20]:
compact = renamed.drop(columns=['IMDB Votes'])
print('Column counts before and after:', renamed.shape[1], compact.shape[1])


Column counts before and after: 12 11


By contrast, `df.drop(index=[104])` removes a row **labeled** 104, not the row at position 104. For removal based on values, a mask describing the rows to keep is usually clearer.

### Update the Intended DataFrame in One Step


In [21]:
flags = screenings.copy()
flags['high_rating'] = False
flags.loc[flags['Rating'] >= 8, 'high_rating'] = True
flags


,Title,Rating,Tickets,high_rating
104,Cedar,7.2,120,False
101,Harbor,8.1,80,True
107,Orbit,6.5,150,False
103,Meadow,8.1,90,True
109,Ember,7.8,110,False


This names the target DataFrame, target rows, and target column together. Avoid chained assignment such as `flags[mask]['high_rating'] = True`; it does not reliably express an update to `flags`, and under pandas 3 Copy-on-Write it does not update the original table.

When you want an independently editable subset, make that choice explicit:

```python
mask = movies['IMDB Rating'] >= 8
subset = movies.loc[mask, ['Title', 'IMDB Rating']].copy()
```

Do not use `other = movies` to make a copy: that gives the same object another name.

### What Does `inplace=True` Mean?

Some methods offer `inplace=True`, which changes the receiving object and returns `None`. It is not a general promise of lower memory use or faster execution. For this course, prefer assigning returned results until you have a reason to use another style.


In [22]:
rename_demo = screenings.copy()
returned = rename_demo.rename(columns={'Tickets': 'tickets_sold'}, inplace=True)
print('Return value:', returned)
print('Columns after the operation:', rename_demo.columns.tolist())


Return value: None
Columns after the operation: ['Title', 'Rating', 'tickets_sold']


Do not write `rename_demo = rename_demo.rename(..., inplace=True)`: that assigns `None` to the variable. Here, checking the returned value and column names is more informative than printing object memory addresses.

## Convert Text and Dates with Checks {#convert-text-and-dates}

Reading Data introduced type inspection. Here we convert columns because a specific operation requires it, and check what the conversion changed. Text dtype names can vary between pandas versions; `object` is not the only way text can be represented.

### Convert Numbers Stored as Text


In [23]:
raw_counts = pd.Series(['1,200', ' 850 ', 'unknown', None], name='followers')
cleaned_counts = raw_counts.str.strip().str.replace(',', '', regex=False)
counts = pd.to_numeric(cleaned_counts, errors='coerce')
conversion_failed = raw_counts.notna() & counts.isna()
print('Converted values:')
print(counts)
print('Non-missing inputs that failed conversion:')
print(raw_counts.loc[conversion_failed])


Converted values:
0    1200.0
1     850.0
2       NaN
3       NaN
Name: followers, dtype: float64
Non-missing inputs that failed conversion:
2    unknown
Name: followers, dtype: str


`errors='coerce'` creates missing values for inputs it cannot convert. It does not repair the source information. The audit distinguishes an originally missing value from an observed string such as `unknown` that failed conversion. Check the failed values before calculating summaries.

Use `.astype('string')` to request a string dtype. `.to_string()` instead produces a formatted textual display; it is not the equivalent dtype-conversion method.

### Apply String Methods to a Column


In [24]:
love_in_title = movies['Title'].str.contains('Love', case=False, regex=False, na=False)
movies.loc[love_in_title, ['Title', 'IMDB Rating']].head(5)


,Title,IMDB Rating
39,Love and Death on Long Island,6.9
48,My Summer of Love,7.0
71,The Love Letter,5.1
124,"I Love You, Beth Cooper",5.9
158,Beloved,5.3


Here, `regex=False` requests a literal substring, and `na=False` treats missing titles as not matching. It finds the text `Love` anywhere in a title, not necessarily as a whole word. Methods such as `.str.strip()`, `.str.lower()`, and `.str.replace()` operate on the values of a Series. Regular expressions belong in later work.

### Parse Dates, Then Use `.dt`


In [25]:
raw_dates = movies['Release Date']
release_dates = pd.to_datetime(raw_dates, format='%b %d %Y', errors='coerce')
failed_dates = raw_dates.notna() & release_dates.isna()
print('Non-missing dates that failed conversion:', failed_dates.sum())
print('Examples:', raw_dates.loc[failed_dates].head().tolist())
release_dates.head(3)


Non-missing dates that failed conversion: 0
Examples: []


0   2006-11-22
1   1965-04-07
2   2009-04-24
Name: Release Date, dtype: datetime64[us]

The declared format matches text such as `Nov 22 2006`. A date that fails conversion becomes `NaT`, pandas' missing datetime value. If a source uses more than one format, inspect it before choosing a parsing strategy.


In [26]:
movie_dates = movies[['Title']].copy()
movie_dates['release_year'] = release_dates.dt.year
reference_date = pd.Timestamp('2024-01-01')
movie_dates['days_since_release'] = (reference_date - release_dates).dt.days
movie_dates.head(3)


,Title,release_year,days_since_release
0,Opal Dreams,2006,6249
1,Major Dundee,1965,21453
2,The Informers,2009,5365


The fixed reference date makes the result reproducible. This is elapsed time **as of January 1, 2024**, not as of the day you run the notebook. `.dt.year` and `.dt.month` give date components; detailed calendar features can wait until later chapters.

## Summarize Values and Define Denominators {#summaries-and-denominators}

### Summarize a Variable You Can Interpret


In [27]:
print('Median IMDB rating:', movies['IMDB Rating'].median())
print('Mean worldwide gross:', movies['Worldwide Gross'].mean())
print('Observed rating count:', movies['IMDB Rating'].count())


Median IMDB rating: 6.4
Mean worldwide gross: 101937019.3016158
Observed rating count: 2228


These summaries answer different questions and have different units. Do not average an IMDB rating and a vote count across a row: although Python can calculate the number, the result has no useful common unit. The mean and median alone also cannot establish a distribution's shape; examine a plot when that is the question.

### Understand `axis` with Comparable Measurements


In [28]:
quarterly_sales = pd.DataFrame(
    {'Q1': [100, 150], 'Q2': [120, 170], 'Q3': [110, 160]},
    index=['North', 'South'],
)
quarterly_sales


,Q1,Q2,Q3
North,100,120,110
South,150,170,160


In [29]:
print('Total per quarter across regions:')
print(quarterly_sales.sum(axis=0))
print('\nTotal per region across quarters:')
print(quarterly_sales.sum(axis=1))


Total per quarter across regions:
Q1    250
Q2    290
Q3    270
dtype: int64

Total per region across quarters:
North    330
South    480
dtype: int64


All entries are sales in the same unit. `axis=0` reduces across the row axis, leaving one answer per column. `axis=1` reduces across the column axis, leaving one answer per row. This prepares you for axes in NumPy without requiring arrays yet.

### Proportions Need an Eligible Group

**Question:** Among movie records with an observed rating, what proportion have a rating of at least 8?


In [30]:
observed_rating = movies['IMDB Rating'].notna()
rated_movies = movies.loc[observed_rating]
numerator = (rated_movies['IMDB Rating'] >= 8).sum()
denominator = len(rated_movies)
proportion = numerator / denominator if denominator > 0 else None
print('Numerator:', numerator)
print('Denominator:', denominator)
print('Proportion:', proportion)


Numerator: 134
Denominator: 2228
Proportion: 0.06014362657091562


Summing a Boolean Series counts `True` values. `len(table)` counts rows, while `series.count()` counts non-missing values. Excluding missing ratings answers a question about **observed ratings**; dividing by all records would answer a different question. If the eligible group is empty, the proportion is undefined rather than zero.

For a conditional proportion, build the subgroup first, then decide which records have the observations needed for the question. Report both numerator and denominator alongside the proportion.

### Count Categories


In [31]:
print('Movie records by genre:')
print(movies['Major Genre'].value_counts())
print('\nProportions among non-missing genres:')
print(movies['Major Genre'].value_counts(normalize=True))


Movie records by genre:
Major Genre
Comedy              685
Drama               613
Action/Adventure    505
Horror/Thriller     340
Western/Musical      54
Documentary          31
Name: count, dtype: int64

Proportions among non-missing genres:
Major Genre
Comedy              0.307451
Drama               0.275135
Action/Adventure    0.226661
Horror/Thriller     0.152603
Western/Musical     0.024237
Documentary         0.013914
Name: proportion, dtype: float64


`value_counts()` excludes missing values by default. `normalize=True` divides by the count of included values. Use `dropna=False` if you want missing entries displayed as a category, and explain that choice. These summaries describe this historical course dataset, not all movies.

## Optional Extensions {#optional-extensions}

These are reference examples, not prerequisites for the in-class activity.

- **Ranks and ties:** `movies['IMDB Rating'].rank(ascending=False, method='min')` assigns tied values the smallest rank in their tied group. Ranking assigns values; sorting changes row order. Try a small tied example before interpreting percentile ranks.
- **Custom sorting:** `sort_values(key=...)` transforms values for sorting. Return to it after learning functions and lambda expressions in more detail.
- **Extreme positions:** `Series.argmax()` returns a position, whereas `idxmax()` returns a label. The NumPy chapter will provide more practice with positions.
- **More date features:** Explore `.dt.quarter` or `.dt.day_name()` after checking the date conversion. ISO week numbers are available through `.dt.isocalendar().week`.

Performance depends on the task, data, and implementation. Choose clear, correct operations first; benchmark a real bottleneck when necessary.

## Independent Practice {#independent-practice}

Use a separate notebook beside `data/`. These longer exercises are additional practice, not requirements for the Chapter 4 quiz. Include your code, outputs, and explanations.

### Album Sales

Read `data/Top 10 Albums By Year.csv`; treat each row as an album entry for a year rather than assuming each title is unique.

1. Inspect `Worldwide Sales` and its dtype. Convert it with `pd.to_numeric(..., errors='coerce')`. Report how many non-missing inputs become missing and show those original values.
2. Assume sales are recorded as of 2022. Create `mean_sales_per_year = Worldwide Sales / (2022 - Year)`. Check that the elapsed-year denominator is positive. Explain that this is a simplified average, not an observed annual sales history.
3. Find the album and artist with the largest observed worldwide sales. State how your method handles ties.
4. Filter `Genre == 'Hip Hop'`. Create `mean_sales_per_year_per_track` by dividing the annualized value by a positive `Tracks` count. **Find the maximum of this new per-track metric**, then report the album, artist, and value.
5. Explain why selecting the largest annualized sales first and only then dividing by tracks does not generally identify the largest per-track value.

### Survey Responses

Read `data/STAT303-1 survey for data analysis.csv`. Inspect the header before renaming. This historical file already uses short names for many variables. Its parties question has a long name; use the following mapping after inspecting it:

```python
survey = pd.read_csv(Path('data') / 'STAT303-1 survey for data analysis.csv')
survey = survey.rename(columns={
    'On average (approx.) how many parties a month do you attend during the school year? Enter a whole number (0, 1, 2, 3, 4, ...)': 'parties_per_month'
})
```

1. Convert `parties_per_month` to numeric and audit conversion failures. Among respondents with an observed numeric answer, calculate the proportion reporting more than four parties per month.
2. Within that subgroup, calculate the proportion whose `introvert_extrovert` answer is `Introvert`, excluding missing answers to that question. Show both counts; inspect category labels before filtering.
3. Use `value_counts(normalize=True)` on `how_happy`. Explain its missing-value treatment. Then find the proportion answering either `Pretty happy` or `Very happy` within the more-than-four-parties subgroup, again using observed answers to that question as the denominator.
4. Preserve the original `num_insta_followers` values. Remove literal commas and tildes with `.str.replace(..., regex=False)`, convert to numeric, and audit failures. Explain why removing `~` retains an approximate number without making it exact.
5. Among respondents with observed numeric follower counts, sort descending. Use `kind='stable'` and retain the original row order for ties. Compare the mean observed `internet_hours_per_day` for the first 46 respondents with the remaining respondents. Report the number of non-missing internet-hour values contributing to each mean. Do not place respondents with unknown follower counts in the lower-follower group.
6. Explain why this descriptive comparison does not show that follower counts cause a difference in internet use.

## Practice Activity: Select, Transform, and Explain {#practice-activity-select-transform-and-explain}

**Goal:** Use pandas to answer questions about rows, columns, and calculated values. **Time:** approximately 30–40 minutes. **File:** `activity04.ipynb` from the [practice kit](https://lizhen0909.github.io/nu-stat303-1-sec20-coursebook-clean/downloads/pandas-fundamentals-practice.zip). **Submit:** `activity04.html` through the final upload question in the Chapter 4 Canvas quiz.

**This section contains the complete activity instructions.** The starter notebook provides places for your work; the independent-practice exercises and optional extensions are not required for this quiz. Work in the extracted `stat303-pandas-fundamentals` folder with the verified project environment.

### A. Explain Labels and Positions

- Replace `Your Name` in the opening Raw cell and Markdown name field. Run the supplied imports and file check; it must report `True`.
- Run the supplied five-row `screenings` table. Before running selections, predict the titles returned by `screenings.loc[[101, 103], 'Title']` and `screenings.iloc[1:3, 0]` in Markdown.
- Run both expressions and explain why their second selected titles differ. Display the shapes of `screenings['Rating']` and `screenings[['Rating']]`, and explain the difference.

### B. Filter and Sort Movie Records

- Read `data/movie_ratings.csv` into `movies` and display its shape.
- Build a mask selecting movies with `IMDB Rating >= 8` **and** `Production Budget < 50000000`. Use parentheses and `&`.
- Use `.loc` to retain those rows and the columns `Title`, `IMDB Rating`, `IMDB Votes`, `Worldwide Gross`, and `Production Budget`. Store an explicit copy as `selected_movies`.
- Report the number selected. Sort by `IMDB Rating` descending, then `IMDB Votes` descending, then `Title` ascending. Display the first five rows and explain the tie-breaking order.

### C. Create a Variable and Keep an Update

- Add `gross_minus_budget_millions` to `selected_movies` using worldwide gross minus production budget, divided by 1,000,000. Display five titles with their calculated values.
- Explain the units, what a negative value means, and why this is not actual profit.
- Rename the calculated column to `gross_budget_gap_millions`, assigning the result to `report`. Display both tables' column names and explain which table has the new name. Explain what would happen if you called `rename()` without keeping its returned result.

### D. Summarize and Explain Your Denominator

- From the **full original `movies` table**, keep the records with observed `IMDB Rating`. Count the records with rating at least 8 and the total with observed ratings. Display both counts and their ratio, guarding against a zero denominator.
- Explain why this denominator differs from the number of movies selected in B and how missing ratings are treated.
- Report the median `gross_budget_gap_millions` in `report` and interpret it for the selected movie records, using the correct units. Do not generalize it to all movies.

### Render and Submit

Restart the kernel, run all cells in order, resolve errors, and save. Add a short Markdown completion note, then save again. From `stat303-pandas-fundamentals` in the terminal, run:

```text
quarto render activity04.ipynb --to html
```

Follow the [Quarto refresher](https://lizhen0909.github.io/nu-stat303-1-sec20-coursebook-clean/vscode_setup.html#render-and-submit-with-quarto): inspect the HTML and a copy opened outside the project folder. Check your name, predictions, code, outputs, and explanations for A–D. Upload only `activity04.html` to the Chapter 4 Canvas quiz; keep your notebook and data locally.

**HTML grading (16 points):** labels, positions, and object shapes (3); filtering, selected columns, and sorting (4); calculated variable and retained rename (4); summaries and denominators (4); name, readable report, and completion note (1).

## Before You Move On {#before-you-move-on}

You should now be able to explain which rows and columns an expression selects, whether a transformation changes the original object, and what a calculated value means. Keep checking units, missing values, tie rules, and denominators alongside your code.

[NumPy Fundamentals](https://lizhen0909.github.io/nu-stat303-1-sec20-coursebook-clean/Numpy.html) comes next. You will reuse comparisons, elementwise arithmetic, and axes with arrays. Pay attention to the shift from pandas' row and column labels to NumPy's positions and shape rules. Pandas Intermediate follows NumPy Fundamentals with more work on alignment and transformations.

References: [pandas indexing](https://pandas.pydata.org/docs/user_guide/indexing.html), [sorting](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sort_values.html), [Copy-on-Write](https://pandas.pydata.org/docs/user_guide/copy_on_write.html), [numeric conversion](https://pandas.pydata.org/docs/reference/api/pandas.to_numeric.html), [datetime conversion](https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html), and [category counts](https://pandas.pydata.org/docs/reference/api/pandas.Series.value_counts.html).
